# Explore how various datasets are geographically related and how to best link various datasets

## Start with visualizing the snapped points (created from running snap_features.slurm)

In [ ]:
import geopandas as gpd

snapped_path = "../../data/processed/snapped_features_19020401.gpkg"

print(gpd.list_layers(snapped_path))

for layer in gpd.list_layers(snapped_path)["name"]:
    gdf = gpd.read_file(snapped_path, layer=layer)
    print(f"\n{layer}: {len(gdf)} features")
    print("CRS:", gdf.crs)

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

# --------------------------------------------------
# 1. File paths
# --------------------------------------------------
huc8 = "19020401"

wbd_path = "../../data/raw/wbd/WBD_National_GPKG.gpkg"
streams_path = "../../data/processed/nhdplus_extracted/NHDPLUS_H_19020401_HU8_GDB/NHDPLUS_H_19020401_HU8_GDB.gdb"
snapped_path = f"../../data/processed/snapped_features_{huc8}.gpkg"

# --------------------------------------------------
# 2. Load watershed
# --------------------------------------------------
layers = gpd.list_layers(wbd_path)
huc8_layer = layers.loc[
    layers["name"].str.contains("hu8", case=False), "name"
].iloc[0]

wbd = gpd.read_file(wbd_path, layer=huc8_layer)

huc_col = next(
    col for col in wbd.columns
    if col.lower() == "huc8"
)

watershed = wbd[
    wbd[huc_col].astype(str).str.zfill(8) == huc8
].copy()

if watershed.empty:
    raise ValueError(f"Could not find HUC8 {huc8} in {huc8_layer}")

watershed_wgs84 = watershed.to_crs("EPSG:4326")

print("Watershed bounds (longitude/latitude):", watershed_wgs84.total_bounds)
print("Estimated UTM CRS:", watershed_wgs84.estimate_utm_crs())

# --------------------------------------------------
# 3. Load stream network
# --------------------------------------------------
stream_layers = gpd.list_layers(streams_path)
streams = gpd.read_file(streams_path, layer="NHDFlowline")

# Clip to the selected watershed.
streams_wgs84 = streams.to_crs("EPSG:4326")
streams_wgs84 = gpd.clip(streams_wgs84, watershed_wgs84)

print("Stream reaches in watershed:", len(streams_wgs84))

# --------------------------------------------------
# 4. Load snapped features
# --------------------------------------------------
snapped_layers = gpd.list_layers(snapped_path)
print("Snapped layers:", snapped_layers["name"].tolist())

stations_layer = next(
    name for name in snapped_layers["name"]
    if "station" in name.lower()
)
dams_layer = next(
    name for name in snapped_layers["name"]
    if "dam" in name.lower()
)

stations_snapped = gpd.read_file(snapped_path, layer=stations_layer)
dams_snapped = gpd.read_file(snapped_path, layer=dams_layer)

# Ensure all layers use longitude/latitude for Folium.
stations_snapped_wgs84 = stations_snapped.to_crs("EPSG:4326")
dams_snapped_wgs84 = dams_snapped.to_crs("EPSG:4326")

# --------------------------------------------------
# 5. Build map
# --------------------------------------------------
center = watershed_wgs84.geometry.union_all().centroid

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=10,
    tiles="OpenStreetMap",
)

# Convert datetime columns to strings for JSON serialization.
watershed_map = watershed_wgs84.copy()

for col in watershed_map.columns:
    if gpd.pd.api.types.is_datetime64_any_dtype(watershed_map[col]):
        watershed_map[col] = watershed_map[col].astype(str)

folium.GeoJson(
    watershed_map,
    name="HUC8 watershed",
    style_function=lambda feature: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.03,
    },
).add_to(m)

# Convert datetime columns to strings for JSON serialization.
streams_map = streams_wgs84.copy()

for col in streams_map.columns:
    if pd.api.types.is_datetime64_any_dtype(streams_map[col]):
        streams_map[col] = streams_map[col].astype(str)

folium.GeoJson(
    streams_map,
    name="NHDFlowline streams",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 1,
        "opacity": 0.6,
    },
).add_to(m)

# Water-quality stations.
print(f"There are {stations_snapped_wgs84.shape[0]} water quality stations")
station_group = folium.FeatureGroup(
    name="Snapped water-quality stations"
)

for _, row in stations_snapped_wgs84.iterrows():
    point = row.geometry

    if point is None or point.is_empty:
        continue

    distance = row.get("snap_distance_m")
    reach_id = row.get("NHDPlusID", "Unknown")
    station_id = row.get("MonitoringLocationIdentifier", "Unknown")

    folium.CircleMarker(
        location=[point.y, point.x],
        radius=3,
        color="darkgreen",
        fill=True,
        fill_color="darkgreen",
        fill_opacity=0.8,
        weight=1,
        popup=(
            f"Station: {station_id}<br>"
            f"Reach ID: {reach_id}<br>"
            f"Snap distance: {distance} m"
        ),
    ).add_to(station_group)

station_group.add_to(m)

# Dam removals.
dam_group = folium.FeatureGroup(name="Snapped dam removals")

for _, row in dams_snapped_wgs84.iterrows():
    point = row.geometry
    if point is None or point.is_empty:
        continue

    folium.Marker(
        location=[point.y, point.x],
        tooltip="Dam removal",
        popup=(
            f"Reach ID: {row.get('NHDPlusID', 'Unknown')}<br>"
            f"Snap distance: {row.get('snap_distance_m', 'Unknown')} m"
        ),
        icon=folium.Icon(color="black", icon="flag"),
    ).add_to(dam_group)

dam_group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

## Now examine the tables themselves:

In [ ]:
# Stations
stations_snapped.head()

In [ ]:
# Dams
dams_snapped.head()

In [ ]:
# Print stream fields
streams = gpd.read_file(streams_path, layer="NHDFlowline")
print(", ".join([col for col in streams.columns]))

In [ ]:
# Now examine other stream layersn for flow:
import fiona
layers = fiona.listlayers(streams_path)
for layer in layers:
    if "flow" in layer.lower():
        print(layer)

In [ ]:
flow = gpd.read_file(
    streams_path,
    layer="NHDPlusFlow",
)

print(", ".join(flow.columns.tolist()))
flow.head()

It seems like NHDPlusFlow contains the data we need to construct the network (ie, "FromNHDPID", "ToNHDPID", and "NodeNumber"). However, quick examination of the head of the table displays node IDs as floats in scientific notation. Next, we will convert these IDs to 

In [ ]:
print(flow[["FromNHDPID", "ToNHDPID", "NodeNumber"]].iloc[0].to_dict())

print("Direction values:")
print(flow["Direction"].value_counts(dropna=False).head(20))

print("NHDPlusID examples:")
print(streams["NHDPlusID"].head(10).to_list())

In [ ]:
stream_ids = set(streams["NHDPlusID"].dropna().astype("int64"))

from_ids = flow["FromNHDPID"].dropna().astype("int64")
to_ids = flow["ToNHDPID"].dropna().astype("int64")

print("Unique stream IDs:", len(stream_ids))
print("From IDs matching streams:", from_ids.isin(stream_ids).sum(), "/", len(from_ids))
print("To IDs matching streams:", to_ids.isin(stream_ids).sum(), "/", len(to_ids))